In [1]:

import sys
import os

sys.path.append('..')

import pandas as pd
import scripts as kw

save_dir = 'test'
os.makedirs(save_dir, exist_ok=True)

train_df = pd.DataFrame([
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 102,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 103,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 105,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 106,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 4
    },
    
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 103,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 5
    },
])

test_df = pd.DataFrame([
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 5
    },
])

concat_df = pd.concat([train_df, test_df], ignore_index=True)

concat_df.to_csv(os.path.join(save_dir, 'test.csv'), sep=kw.DELIMITER, header=True, index=False, encoding=kw.ENCODING, quoting=kw.QUOTING, quotechar=kw.QUOTECHAR)

In [2]:
import numpy as np

target_embeddings = np.array([
    [1 ,  0],
    [0 ,  1],
    [-1,  0],
    [0 , -1],
    [1 ,  0],
    [0 ,  1],
    [-1,  0],
    [0 , -1],
])

context_embeddings = np.array([
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
])

np.save(os.path.join(save_dir, kw.FILE_ITEMS_EMBEDDINGS), target_embeddings)
np.save(os.path.join(save_dir, kw.FILE_CONTEXT_EMBEDDINGS), context_embeddings)

In [3]:
from scripts.modules.recommenders.Item2vec.Data_repr import DataRepr
import pickle

data_repr = DataRepr(train_df)

with open(os.path.join(save_dir, kw.FILE_SPARSE_REPR), 'wb') as f:
    pickle.dump(data_repr, f)

In [4]:
df = pd.read_csv(os.path.join(save_dir, 'test.csv'), delimiter=kw.DELIMITER, encoding=kw.ENCODING, quoting=kw.QUOTING, quotechar=kw.QUOTECHAR, header=0)
df = df.dropna().drop_duplicates(subset=[kw.COLUMN_USER_ID, kw.COLUMN_ITEM_ID], keep='last')

df

,id_user,id_item,rating
0,99,100,4
1,99,101,4
2,99,102,4
3,99,103,4
4,99,104,4
5,99,105,4
6,99,106,4
7,99,107,4
8,100,100,5
9,100,101,0


In [5]:
df_train = df.loc[:len(train_df)-1]

df_train

,id_user,id_item,rating
0,99,100,4
1,99,101,4
2,99,102,4
3,99,103,4
4,99,104,4
5,99,105,4
6,99,106,4
7,99,107,4
8,100,100,5
9,100,101,0


In [6]:
df_test = df.loc[len(train_df):]
df_test

,id_user,id_item,rating
14,100,104,5
15,100,107,0
16,101,104,5
17,101,107,0
18,102,104,5
19,102,107,5


In [7]:
import torch
from sklearn.metrics.pairwise import cosine_similarity

def combine_embeddings(target_embeddings, context_embeddings, combination_strategy='avg_norm_after', use_norm=True):

    t = torch.from_numpy(target_embeddings).float()
    c = torch.from_numpy(context_embeddings).float()

    def _norm(x):
        return x / (x.norm(dim=1, keepdim=True) + 1e-9)

    if use_norm:
        t = _norm(t)
        c = _norm(c)
        
    combined = t + c

    return combined.numpy()

class ItemSim:

    def __init__(self, embeddings_filepath, use_norm=True, combination_strategy='avg_norm_after', k=kw.K):
        
        self.embedding_dir = embeddings_filepath
        self.use_norm = use_norm
        self.combination_strategy = combination_strategy
        self.k = k

        with open(os.path.join(embeddings_filepath, kw.FILE_SPARSE_REPR), 'rb') as f:
            self.data_repr = pickle.load(f)
        
        target_embeddings = np.load(os.path.join(embeddings_filepath, kw.FILE_ITEMS_EMBEDDINGS))
        context_embeddings = np.load(os.path.join(embeddings_filepath, kw.FILE_CONTEXT_EMBEDDINGS))

        self.item_embeddings = combine_embeddings(
            target_embeddings,
            context_embeddings,
            combination_strategy=self.combination_strategy,
            use_norm=self.use_norm
        )

    def fit(self, df_train):
        # Group by user, get the items index history, and store in a dict
        explicit_ratings = df_train[kw.COLUMN_RATING]!=-1
        min_max = df_train[explicit_ratings][kw.COLUMN_RATING].agg(['min', 'max'])
        mean_rating = (min_max.loc['min'] + min_max.loc['max']) / 2  
        df_train = df_train[(df_train[kw.COLUMN_RATING]>=mean_rating)|(df_train[kw.COLUMN_RATING]==-1)]
        self.user_histories = df_train.groupby(kw.COLUMN_USER_ID)[kw.COLUMN_ITEM_ID].apply(lambda row: self.data_repr.le_items.transform(row.tolist())).to_dict()

        self.max_rating = min_max.loc['max']
        self.min_rating = min_max.loc['min']

        print(self.min_rating, self.max_rating)

    def predict(self, df_test):
        for user_id, item_id in zip(df_test[kw.COLUMN_USER_ID], df_test[kw.COLUMN_ITEM_ID]):
            target_item_embedding = self.item_embeddings[self.data_repr.get_item_index(item_id)]
            user_history_embeddings = self.item_embeddings[self.user_histories[user_id]]
            similarity = cosine_similarity([target_item_embedding], user_history_embeddings).mean()

            print(similarity)

            score = (similarity + 1) / (2) * (self.max_rating - self.min_rating) + self.min_rating
            print(score)

In [8]:
model = ItemSim(save_dir)
model.item_embeddings

array([[ 1.,  0.],
       [ 0.,  1.],
       [-1.,  0.],
       [ 0., -1.],
       [ 1.,  0.],
       [ 0.,  1.],
       [-1.,  0.],
       [ 0., -1.]], dtype=float32)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(model.item_embeddings)

array([[ 1.,  0., -1.,  0.,  1.,  0., -1.,  0.],
       [ 0.,  1.,  0., -1.,  0.,  1.,  0., -1.],
       [-1.,  0.,  1.,  0., -1.,  0.,  1.,  0.],
       [ 0., -1.,  0.,  1.,  0., -1.,  0.,  1.],
       [ 1.,  0., -1.,  0.,  1.,  0., -1.,  0.],
       [ 0.,  1.,  0., -1.,  0.,  1.,  0., -1.],
       [-1.,  0.,  1.,  0., -1.,  0.,  1.,  0.],
       [ 0., -1.,  0.,  1.,  0., -1.,  0.,  1.]], dtype=float32)

In [10]:
model.fit(df_train)
model.user_histories

0 5


{99: array([0, 1, 2, 3, 4, 5, 6, 7]),
 100: array([0]),
 101: array([3]),
 102: array([0, 1])}

In [11]:
cosine_similarity(model.item_embeddings, [[0, 1]])

array([[ 0.],
       [ 1.],
       [ 0.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 0.],
       [-1.]])

In [12]:
model.predict(df_test)

1.0
5.0
0.0
2.5
0.0
2.5
1.0
5.0
0.5
3.75
-0.5
1.25
